# Smale's Problem

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order




def polynomial_roots_to_coeffs(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  return np.polyder(coeffs)


def polynomial_critical_points(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


@numba.njit
def roots_to_coeffs_jit(roots):
  """Calculates polynomial coefficients from roots (JIT-compiled)."""
  coeffs = np.array([1.0 + 0.0j])

  for r in roots:
    coeffs = np.append(coeffs * (-r), 0) + np.append(
        np.array([0.0 + 0.0j]), coeffs
    )
  return coeffs[::-1]  # REVERSE THE COEFFICIENT ARRAY HERE


@numba.njit
def poly_derivative_jit(coeffs):
  """Calculates polynomial derivative coefficients (JIT-compiled) - CORRECTED ORDER."""
  n_plus_one = len(coeffs)  # Number of coefficients (degree + 1)
  degree = n_plus_one - 1  # Degree of polynomial
  deriv_coeffs = np.zeros(degree, dtype=coeffs.dtype)  # Derivative has degree-1

  for i in range(
      degree
  ):  # Iterate from i=0 to degree-1 (for deriv_coeffs indices)
    deriv_coeffs[i] = coeffs[i] * (
        degree - i
    )  # Correct formula: (n-i)*a_{i}*x^(n-1-i)
    # coeffs[i] is a_{n-i}, (degree - i) is the power
  return deriv_coeffs


@numba.njit
def polynomial_val_jit(coeffs, z):
  """Evaluate polynomial using JIT compilation - CORRECTED for NumPy order."""
  result = 0.0 + 0.0j
  for (
      coeff
  ) in coeffs:  # Iterate in DIRECT order (highest degree to constant) - CORRECT
    result = result * z + coeff
  return result


@numba.njit
def evaluate_smale_conjecture_jit(roots, grid_size):
  """Evaluates Smale's conjecture using a grid of test points (JIT-compiled)."""
  n = len(roots)
  if n < 10:
    return 0.0

  coeffs = roots_to_coeffs_jit(roots)
  derivative_coeffs = poly_derivative_jit(coeffs)

  critical_points = np.roots(derivative_coeffs)

  max_ratio = 0.0

  x_vals = np.linspace(-2, 2, grid_size)
  y_vals = np.linspace(-2, 2, grid_size)
  z_grid = np.array([x + 1j * y for x in x_vals for y in y_vals])  # pylint: disable=g-complex-comprehension

  for z in z_grid:
    f_prime_z = polynomial_val_jit(derivative_coeffs, z)
    if np.abs(f_prime_z) < 0.01:
      continue
    if np.abs(f_prime_z) > 10_000_000.0:
      return 0

    min_smale_ratio_for_z = float('inf')

    for xi in critical_points:
      numerator = np.abs(
          polynomial_val_jit(coeffs, z) - polynomial_val_jit(coeffs, xi)
      )
      denominator = np.abs(z - xi) * np.abs(f_prime_z)

      if denominator < 1e-9:
        continue
      smale_ratio = numerator / denominator
      min_smale_ratio_for_z = min(min_smale_ratio_for_z, smale_ratio)

    max_ratio = max(max_ratio, min_smale_ratio_for_z)

  return max_ratio


def evaluate_smale_conjecture(roots, grid_size=10):
  return evaluate_smale_conjecture_jit(roots, grid_size)


############### #Hidden variants of above code


def polynomial_roots_to_coeffs_h(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs_h(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  return np.polyder(coeffs)


def polynomial_critical_points_h(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs_h(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs_h(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


def evaluate_smale_conjecture_h(roots: np.ndarray, grid_size):
  """Evaluates Smale's conjecture using a grid of test points.

  Args:
    roots: Roots of the polynomial.
    grid_size: Number of points along each axis for the grid in [-2, 2] x [-2,
      2].

  Returns:
    Maximum Smale ratio found on the grid.
  """
  n = len(roots)
  if n < 10:
    return 0  # Conjecture is for degree n >= 2

  coeffs = polynomial_roots_to_coeffs(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs(coeffs)
  critical_points = polynomial_critical_points(roots)

  max_ratio = 0.0

  # Create grid of test points in [-2, 2] x [-2, 2]
  x_vals = np.linspace(-2, 2, grid_size)
  y_vals = np.linspace(-2, 2, grid_size)
  z_grid = np.array([x + 1j * y for x in x_vals for y in y_vals])  # pylint: disable=g-complex-comprehension

  for z in z_grid:
    # Ensure f'(z) is not zero
    f_prime_z = np.polyval(derivative_coeffs, z)
    if np.abs(f_prime_z) < 0.01:
      continue
    if np.abs(f_prime_z) > 10_000_000.0:
      return 0

    min_smale_ratio_for_z = float('inf')

    for xi in critical_points:
      numerator = np.abs(np.polyval(coeffs, z) - np.polyval(coeffs, xi))
      denominator = np.abs(z - xi) * np.abs(f_prime_z)

      if np.isclose(denominator, 0):
        continue
      smale_ratio = numerator / denominator
      min_smale_ratio_for_z = min(min_smale_ratio_for_z, smale_ratio)

    max_ratio = max(max_ratio, min_smale_ratio_for_z)

  return max_ratio


def format_feedback_repr(feedback):
  """Formats feedback dictionary for code representation."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for Smale's conjecture, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  best_construction = search_for_best_poly()
  result['score'] = evaluate_smale_conjecture_h(best_construction, 91)

  feedback['best_roots'] = best_construction
  feedback['best_score_found'] = result['score']
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds a function that give the best bound for Smale's conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import numba
import re
from typing import Any, Callable, Mapping
import scipy.linalg as la
import numpy.polynomial.polynomial as poly

minimize = optimize.minimize



def search_for_best_poly() -> np.ndarray:
  """Function to search for the best construction."""
  roots = np.array(
      [
          0,
          7.0538085 - 0.416j,
          6.22735 + 0.742161j,
          0.2409735 - 5.25161j,
          -0.4105 + 4.872849j,
          -0.405 - 3.8472849j,
          -0.71529546 + 2.0j,
          1 + 0.1j,
          0.2 + 0.2j,
          2.2 - 0.3j,
      ],
      dtype=np.complex128,
  )
  best_roots = roots.copy()
  best_score = evaluate_smale_conjecture(roots, 10)
  start_time = time.time()
  eval_count = 0
  while time.time() - start_time < np.random.randint(10, 500):
    roots += np.random.uniform(-0.1, 0.1, size=roots.shape)  # Perturb all roots
    score = evaluate_smale_conjecture(roots, 10)
    eval_count += 1
    if score > best_score:
      best_score = score
      best_roots = roots.copy()
      print(f'Improved Smale score: {score}')
  print(eval_count)
  return best_roots

**Prompt used**

Act as an expert software developer and inequality specialist specializing in creating polynomials with certain properties. You will be trying to find good examples to a complicated analysis problem.
Your task is to generate the sequence of roots of a complex polynomial, that maximizes the following evaluation function:

def evaluate_smale_conjecture(roots, grid_size):
  """Evaluates Smale's conjecture using a grid of test points.

Args:
    roots: Roots of the polynomial.
    grid_size: Number of points along each axis for the grid in [-2, 2] x [-2, 2].

Returns:
    Maximum Smale ratio found on the grid.
  """
  n = len(roots)
  if n < 10:
    return 0  # Conjecture is for degree n >= 2

coeffs = polynomial_roots_to_coeffs(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs(coeffs)
  critical_points = polynomial_critical_points(roots)

max_ratio = 0.0

# Create grid of test points in [-2, 2] x [-2, 2]
  x_vals = np.linspace(-2, 2, grid_size)
  y_vals = np.linspace(-2, 2, grid_size)
  z_grid = np.array([x + 1j * y for x in x_vals for y in y_vals])

for z in z_grid:
    # Ensure f'(z) is not zero
    f_prime_z = np.polyval(derivative_coeffs, z)
    if np.isclose(f_prime_z, 0):
      continue

min_smale_ratio_for_z = float('inf')

for xi in critical_points:
  numerator = np.abs(np.polyval(coeffs, z) - np.polyval(coeffs, xi))
  denominator = np.abs(z - xi) * np.abs(f_prime_z)

  if np.isclose(denominator, 0):
    continue
  smale_ratio = numerator / denominator
  min_smale_ratio_for_z = min(min_smale_ratio_for_z, smale_ratio)

max_ratio = max(max_ratio, min_smale_ratio_for_z)
return max_ratio

Your task is to write a search function that searches for the best list of roots. Your function will have 500 seconds to run, and after that it has to have returned the best construction it found. If after 500 seconds it has not returned anything, it will be terminated with negative infinity points. You may freely choose the value of n, the number of zeros. To find a counterexample where the final score is greater than 1, you will likely have to look in the n ~ 10-12 range. Your n must be at least 10 for it to receive a score.

You may code up any search method you want, and you are allowed to call the evaluate_smale_conjecture() function as many times as you want. You have access to it, you don't need to code up the evaluate_smale_conjecture() function.

## What AlphaEvolve found

AlphaEvolve matched the best known lower bound for $C(n)$ by finding the $z^n - nz$ optimizer, which gives $C(n) \geq 1 - 1/n$. It also found some other constructions with similar score, but did not manage to find a counterexample to Smale's conjecture that $C(n) = 1 - 1/n$.